# Cleaning Data

## Cleaning Data IHSG

In [22]:
# ============================================================
# CELL 1: IMPORT & LOAD IHSG
# ============================================================
import pandas as pd
import numpy as np


# Load data mentah
df_ihsg = pd.read_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\ihsg_2019_2024.csv')
print(f"📈 IHSG mentah: {len(df_ihsg)} baris")
print(df_ihsg.head())

# ============================================================
# CELL 2: CLEANING IHSG
# ============================================================
# Konversi tanggal
df_ihsg['date'] = pd.to_datetime(df_ihsg['date'])

# Sort by date
df_ihsg = df_ihsg.sort_values('date').reset_index(drop=True)

# Handle missing values dari rolling window (normal di awal data)
print(f"\nMissing values sebelum cleaning:")
print(df_ihsg.isnull().sum())

# Forward fill untuk volatility dan MA (lebih baik daripada drop)
df_ihsg['volatility_20d'] = df_ihsg['volatility_20d'].fillna(method='ffill')
df_ihsg['ma_50'] = df_ihsg['ma_50'].fillna(method='ffill')

# Drop baris dengan daily_return/log_return NaN (hanya hari pertama)
df_ihsg = df_ihsg.dropna(subset=['daily_return', 'log_return'])

print(f"\n✅ IHSG setelah cleaning: {len(df_ihsg)} baris")
print(f"Missing values setelah cleaning:")
print(df_ihsg.isnull().sum())

# Simpan sementara
df_ihsg.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\ihsg_cleaned.csv', index=False)
print("\n💾 Disimpan: data/processed/ihsg_cleaned.csv")

📈 IHSG mentah: 1456 baris
         date         open         high          low        close    volume  \
0  2019-01-02  6197.871094  6205.895020  6164.833984  6181.174805  52797800   
1  2019-01-03  6176.151855  6221.009766  6176.151855  6221.009766  72166700   
2  2019-01-04  6211.096191  6274.540039  6200.854004  6274.540039  80858100   
3  2019-01-07  6317.625977  6354.757812  6287.224121  6287.224121  90278300   
4  2019-01-08  6292.263184  6316.240234  6251.375977  6262.847168  90537400   

   daily_return  log_return  volatility_20d  ma_50  
0           NaN         NaN             NaN    NaN  
1      0.006445    0.006424             NaN    NaN  
2      0.008605    0.008568             NaN    NaN  
3      0.002022    0.002019             NaN    NaN  
4     -0.003877   -0.003885             NaN    NaN  

Missing values sebelum cleaning:
date               0
open               0
high               0
low                0
close              0
volume             0
daily_return       1


C:\Users\ADVAN\AppData\Local\Temp\ipykernel_10484\1606743233.py:27: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ihsg['volatility_20d'] = df_ihsg['volatility_20d'].fillna(method='ffill')
C:\Users\ADVAN\AppData\Local\Temp\ipykernel_10484\1606743233.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ihsg['ma_50'] = df_ihsg['ma_50'].fillna(method='ffill')


## Cleaning Data Berita

In [23]:
# ============================================================
# CELL 3: LOAD & CLEANING BERITA
# ============================================================
df_news = pd.read_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\news_googlerss_2019_2024.csv')
print(f"📰 Berita mentah: {len(df_news)} artikel")

# Konversi tanggal
df_news['publish_date'] = pd.to_datetime(df_news['publish_date'], errors='coerce')

# Hapus baris tanpa tanggal atau judul
df_news = df_news.dropna(subset=['publish_date', 'title'])

# Hapus duplikat judul (case-insensitive)
df_news['title_lower'] = df_news['title'].str.lower()
df_news = df_news.drop_duplicates(subset=['title_lower'], keep='first')
df_news = df_news.drop(columns=['title_lower'])

# Hapus judul yang terlalu pendek (< 20 karakter, kemungkinan bukan artikel)
df_news = df_news[df_news['title'].str.len() >= 20]

# Filter hanya artikel 2019–2024
df_news = df_news[(df_news['publish_date'] >= '2019-01-01') & 
                  (df_news['publish_date'] <= '2024-12-31')]

print(f"\n✅ Berita setelah cleaning: {len(df_news)} artikel")
print(f"Per source:\n{df_news['source'].value_counts()}")
print(f"\nRentang tanggal: {df_news['publish_date'].min()} → {df_news['publish_date'].max()}")

# Simpan
df_news.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\news_cleaned.csv', index=False)
print("\n💾 Disimpan: data/processed/news_cleaned.csv")

📰 Berita mentah: 1905 artikel

✅ Berita setelah cleaning: 1905 artikel
Per source:
source
CNBC Indonesia           374
kompas.id                 99
kontan.co.id              88
detikFinance              64
Liputan6.com              61
                        ... 
Fajarpos Network           1
indoposco.id               1
PelitaRiau.Com             1
Indonesia Investments      1
Rmol.id                    1
Name: count, Length: 211, dtype: int64

Rentang tanggal: 2019-01-02 00:00:00 → 2024-12-31 00:00:00

💾 Disimpan: data/processed/news_cleaned.csv


# TEXT PREPROCESSING

In [24]:
# ============================================================
# CELL 4: TEXT PREPROCESSING
# ============================================================
import re
import string

# Install Sastrawi kalau belum
# !pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def clean_text(text):
    """
    Pipeline preprocessing teks bahasa Indonesia
    """
    if pd.isna(text):
        return ""
    
    # 1. Case folding
    text = text.lower()
    
    # 2. Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Hapus mention, hashtag, email
    text = re.sub(r'@\w+|#\w+|\S+@\S+', '', text)
    
    # 4. Hapus angka dan tanda baca
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 5. Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 6. Stopword removal
    text = stopword_remover.remove(text)
    
    # 7. Stemming (opsional, bisa di-skip untuk IndoBERT)
    # text = stemmer.stem(text)
    
    return text

# Terapkan ke judul berita
print("⏳ Preprocessing teks berita...")
df_news['title_clean'] = df_news['title'].apply(clean_text)

print("✅ Contoh hasil preprocessing:")
for i in range(3):
    print(f"   Asli:  {df_news['title'].iloc[i]}")
    print(f"   Bersih: {df_news['title_clean'].iloc[i]}")
    print()

# Simpan
df_news.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\news_cleaned.csv', index=False)

⏳ Preprocessing teks berita...
✅ Contoh hasil preprocessing:
   Asli:  IHSG ditutup turun 31,54 poin, dipicu sentimen negatif global - antaranews.com
   Bersih: ihsg ditutup turun poin dipicu sentimen negatif global antaranewscom

   Asli:  Perdagangan Saham Terakhir di 2019 Berada di Zona Merah, BEI: Kita Masih yang Terbaik di ASEAN - Voice of America Indonesia
   Bersih: perdagangan saham terakhir berada zona merah bei masih terbaik asean voice of america indonesia

   Asli:  IHSG Tertekan, Sabar! Ini Nasihat BEI Berinvestasi Saham - CNBC Indonesia
   Bersih: ihsg tertekan sabar nasihat bei berinvestasi saham cnbc indonesia



# SENTIMENT ANALYSIS - VADER (Baseline)

In [25]:
# ============================================================
# CELL 5: SENTIMENT ANALYSIS - VADER (Baseline)
# ============================================================
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    """
    VADER pada teks Indonesia (judul berita)
    """
    if not text or pd.isna(text):
        return 0, 'neutral'
    
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    
    # Klasifikasi
    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'
    
    return compound, label

print("⏳ Analisis sentimen VADER...")
df_news[['vader_score', 'vader_label']] = df_news['title_clean'].apply(
    lambda x: pd.Series(get_vader_sentiment(x))
)

print("✅ Contoh hasil VADER:")
print(df_news[['title', 'vader_score', 'vader_label']].head(10))

print(f"\nDistribusi sentimen VADER:")
print(df_news['vader_label'].value_counts())

# Simpan
df_news.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\news_with_sentiment.csv', index=False)
print("\n💾 Disimpan: data/processed/news_with_sentiment.csv")

⏳ Analisis sentimen VADER...
✅ Contoh hasil VADER:
                                               title  vader_score vader_label
0  IHSG ditutup turun 31,54 poin, dipicu sentimen...          0.0     neutral
1  Perdagangan Saham Terakhir di 2019 Berada di Z...          0.0     neutral
2  IHSG Tertekan, Sabar! Ini Nasihat BEI Berinves...          0.0     neutral
3  IHSG mampu menguat di bulan Oktober 2019, baga...          0.0     neutral
4  Saham Unilever Naik 57% dalam 5 Tahun, Begini ...          0.0     neutral
5  Penutupan IHSG Akhir Tahun, Harapan Sri Mulyan...          0.0     neutral
6  Asing Beli Bersih sampai Rp9 Triliun, IHSG Han...          0.0     neutral
7  IHSG Ditutup Hijau, Menguat ke Level 6.326 - k...          0.0     neutral
8  IHSG Ambruk, Ini Rekomendasi Lo Kheng Hong! - ...          0.0     neutral
9  Sembilan Saham Hijau, Ini 10 Saham LQ45 dengan...          0.0     neutral

Distribusi sentimen VADER:
vader_label
neutral     1829
positive      53
negative      23


# AGREGASI SENTIMEN HARIAN

In [26]:
# ============================================================
# CELL 6: AGREGASI SENTIMEN HARIAN
# ============================================================

# Agregasi per tanggal
daily_sentiment = df_news.groupby('publish_date').agg({
    'vader_score': ['mean', 'std', 'count'],
    'vader_label': lambda x: (x == 'positive').sum() / len(x)
}).reset_index()

# Flatten nama kolom
daily_sentiment.columns = ['date', 'avg_sentiment', 'std_sentiment', 'news_count', 'positive_ratio']

# Isi hari tanpa berita dengan 0 (artinya netral/tidak ada berita)
date_range = pd.date_range(start='2019-01-01', end='2024-12-31', freq='D')
daily_sentiment = daily_sentiment.set_index('date').reindex(date_range).reset_index()
daily_sentiment = daily_sentiment.rename(columns={'index': 'date'})

# Fill NaN
daily_sentiment['avg_sentiment'] = daily_sentiment['avg_sentiment'].fillna(0)
daily_sentiment['std_sentiment'] = daily_sentiment['std_sentiment'].fillna(0)
daily_sentiment['news_count'] = daily_sentiment['news_count'].fillna(0).astype(int)
daily_sentiment['positive_ratio'] = daily_sentiment['positive_ratio'].fillna(0)

print(f"✅ Daily Sentiment Index: {len(daily_sentiment)} hari")
print(daily_sentiment.head(10))
print(f"\nStatistik:")
print(daily_sentiment[['avg_sentiment', 'news_count']].describe())

# Simpan
daily_sentiment.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\daily_sentiment_index.csv', index=False)
print("\n💾 Disimpan: data/processed/daily_sentiment_index.csv")

✅ Daily Sentiment Index: 2192 hari
        date  avg_sentiment  std_sentiment  news_count  positive_ratio
0 2019-01-01            0.0            0.0           0             0.0
1 2019-01-02            0.0            0.0           2             0.0
2 2019-01-03            0.0            0.0           1             0.0
3 2019-01-04            0.0            0.0           0             0.0
4 2019-01-05            0.0            0.0           0             0.0
5 2019-01-06            0.0            0.0           0             0.0
6 2019-01-07            0.0            0.0           0             0.0
7 2019-01-08            0.0            0.0           0             0.0
8 2019-01-09            0.0            0.0           1             0.0
9 2019-01-10            0.0            0.0           0             0.0

Statistik:
       avg_sentiment   news_count
count    2192.000000  2192.000000
mean        0.002636     0.869069
std         0.049987     1.261627
min        -0.585900     0.000000
25

# Merge IHSG + Sentimen

In [27]:
# ============================================================
# CELL 7: MERGE DATASET MASTER
# ============================================================

# Load
df_ihsg = pd.read_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\ihsg_cleaned.csv')
df_sentiment = pd.read_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\daily_sentiment_index.csv')

# Pastikan tanggal sama-sama datetime
df_ihsg['date'] = pd.to_datetime(df_ihsg['date'])
df_sentiment['date'] = pd.to_datetime(df_sentiment['date'])

# Merge (left join: semua hari IHSG, tambahkan sentimen kalau ada)
df_master = pd.merge(df_ihsg, df_sentiment, on='date', how='left')

# Fill NaN untuk hari libur tanpa berita
df_master['avg_sentiment'] = df_master['avg_sentiment'].fillna(0)
df_master['std_sentiment'] = df_master['std_sentiment'].fillna(0)
df_master['news_count'] = df_master['news_count'].fillna(0).astype(int)
df_master['positive_ratio'] = df_master['positive_ratio'].fillna(0)

print(f"✅ Master Dataset: {len(df_master)} baris")
print(f"Kolom: {list(df_master.columns)}")
print(df_master.head())

# ============================================================
# CELL 8: FEATURE ENGINEERING (LAG & ROLLING)
# ============================================================

# Lag variables (sentimen hari sebelumnya prediksi hari ini)
for lag in [1, 2, 3]:
    df_master[f'sentiment_lag_{lag}'] = df_master['avg_sentiment'].shift(lag)

# Rolling average sentimen (7 hari)
df_master['sentiment_ma_7'] = df_master['avg_sentiment'].rolling(window=7).mean()

# Rolling average sentimen (14 hari)
df_master['sentiment_ma_14'] = df_master['avg_sentiment'].rolling(window=14).mean()

# Sentiment momentum (perubahan sentimen dari kemarin)
df_master['sentiment_change'] = df_master['avg_sentiment'] - df_master['sentiment_lag_1']

# Drop baris dengan NaN akibat lag/rolling
df_master = df_master.dropna()

print(f"\n✅ Master Dataset final: {len(df_master)} baris")
print(f"Kolom final: {list(df_master.columns)}")
print(df_master.tail())

# Simpan
df_master.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\master_dataset.csv', index=False)
print("\n💾 MASTER DATASET: data/processed/master_dataset.csv")

✅ Master Dataset: 1455 baris
Kolom: ['date', 'open', 'high', 'low', 'close', 'volume', 'daily_return', 'log_return', 'volatility_20d', 'ma_50', 'avg_sentiment', 'std_sentiment', 'news_count', 'positive_ratio']
        date         open         high          low        close     volume  \
0 2019-01-03  6176.151855  6221.009766  6176.151855  6221.009766   72166700   
1 2019-01-04  6211.096191  6274.540039  6200.854004  6274.540039   80858100   
2 2019-01-07  6317.625977  6354.757812  6287.224121  6287.224121   90278300   
3 2019-01-08  6292.263184  6316.240234  6251.375977  6262.847168   90537400   
4 2019-01-09  6296.115234  6311.579102  6265.326172  6272.237793  105604200   

   daily_return  log_return  volatility_20d  ma_50  avg_sentiment  \
0      0.006445    0.006424             NaN    NaN            0.0   
1      0.008605    0.008568             NaN    NaN            0.0   
2      0.002022    0.002019             NaN    NaN            0.0   
3     -0.003877   -0.003885            

# VERIFIKASI AKHIR


In [28]:
# ============================================================
# CELL 9: VERIFIKASI AKHIR FASE 3
# ============================================================

import os


print("=" * 60)
print("📊 VERIFIKASI FASE 3: DATA PREPARATION")
print("=" * 60)

files = {
    'ihsg_cleaned': r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\ihsg_cleaned.csv',
    'news_cleaned': r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\news_cleaned.csv',
    'daily_sentiment': r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\daily_sentiment_index.csv',
    'master': r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\master_dataset.csv'
}

for name, path in files.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"\n✅ {name}: {len(df)} baris | {path}")
    else:
        print(f"\n❌ {name}: TIDAK DITEMUKAN")

print("\n" + "=" * 60)
print("CHECKLIST FASE 3:")
print("=" * 60)

checks = [
    (os.path.exists(files['master']), "Master dataset tersedia"),
    (len(pd.read_csv(files['master'])) > 1000, "Master > 1000 baris"),
    ('sentiment_lag_1' in pd.read_csv(files['master']).columns, "Ada lag features"),
    ('sentiment_ma_7' in pd.read_csv(files['master']).columns, "Ada rolling features"),
]

for ok, msg in checks:
    status = "✅" if ok else "❌"
    print(f"   {status} {msg}")

print("=" * 60)

📊 VERIFIKASI FASE 3: DATA PREPARATION

✅ ihsg_cleaned: 1455 baris | C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\ihsg_cleaned.csv

✅ news_cleaned: 1905 baris | C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\news_cleaned.csv

✅ daily_sentiment: 2192 baris | C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\daily_sentiment_index.csv

✅ master: 1407 baris | C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\processed\master_dataset.csv

CHECKLIST FASE 3:
   ✅ Master dataset tersedia
   ✅ Master > 1000 baris
   ✅ Ada lag features
   ✅ Ada rolling features
